In [1]:
from collections import defaultdict, deque
from pathlib import Path

import pandas as pd
from IPython.display import display
from rdflib import Graph, URIRef
from rdflib.namespace import RDFS

In [2]:
BASE_DIR   = Path(".").resolve()
ONTOLOGY   = BASE_DIR / "ontology_classes_reasoned.owl"
QUERY_FILE = BASE_DIR / "ofl_subclasses_of_result_fma_superclasses.sparql"
OUTPUT_CSV = BASE_DIR / "ofl_2.1.0_subclasses_of_result_fma_superclasses.csv"

OFL_PREFIX = "https://purl.bioontology.org/ontology/OFL/"

In [3]:
graph = Graph()
graph.parse(ONTOLOGY, format="xml")
print(f"Loaded {len(graph):,} triples from {ONTOLOGY.name}")

Loaded 14,502 triples from ontology_classes_reasoned.owl


In [4]:
# Step 1 — find root OFL classes (fast: no transitive path in SPARQL)
query = QUERY_FILE.read_text(encoding="utf-8")
roots_result = graph.query(query)

roots = {
    str(row.root): {
        "label":          str(row.rootLabel) if row.rootLabel else "",
        "fmaSuperclasses": str(row.fmaSuperclasses) if row.fmaSuperclasses else "",
    }
    for row in roots_result
}
print(f"Found {len(roots)} root OFL classes")

Found 15 root OFL classes


In [5]:
# Step 2 — build direct-subclass index from rdflib triples (fast)
children: dict[str, list[str]] = defaultdict(list)
for s, _, o in graph.triples((None, RDFS.subClassOf, None)):
    if isinstance(s, URIRef) and isinstance(o, URIRef):
        if str(s).startswith(OFL_PREFIX):
            children[str(o)].append(str(s))

# Step 3 — BFS from each root to collect all OFL descendants
def label_of(iri: str) -> str:
    for _, _, lbl in graph.triples((URIRef(iri), RDFS.label, None)):
        return str(lbl)
    return ""

rows = []
for root_iri, root_meta in roots.items():
    queue = deque([root_iri])
    visited: set[str] = set()
    while queue:
        current = queue.popleft()
        if current in visited:
            continue
        visited.add(current)
        rows.append({
            "class":          current,
            "classLabel":     label_of(current),
            "root":           root_iri,
            "rootLabel":      root_meta["label"],
            "fmaSuperclasses": root_meta["fmaSuperclasses"],
        })
        queue.extend(children.get(current, []))

df = pd.DataFrame(rows, columns=["class", "classLabel", "root", "rootLabel", "fmaSuperclasses"])
df.sort_values(["root", "class"], inplace=True, ignore_index=True)
print(f"{len(df)} rows ({df['root'].nunique()} roots)")
display(df)

873 rows (15 roots)


,class,classLabel,root,rootLabel,fmaSuperclasses
0,https://purl.bioontology.org/ontology/OFL/OFLI...,Parascapular cutaneous artery,https://purl.bioontology.org/ontology/OFL/OFLI...,Parascapular cutaneous artery,http://purl.obolibrary.org/obo/FMA_23193
1,https://purl.bioontology.org/ontology/OFL/OFLI...,Scapular cutaneous artery,https://purl.bioontology.org/ontology/OFL/OFLI...,Scapular cutaneous artery,http://purl.obolibrary.org/obo/FMA_23193
2,https://purl.bioontology.org/ontology/OFL/OFLI...,Ascending branch of circumflex scapular artery,https://purl.bioontology.org/ontology/OFL/OFLI...,Ascending branch of circumflex scapular artery,http://purl.obolibrary.org/obo/FMA_23193
3,https://purl.bioontology.org/ontology/OFL/OFLI...,Intrafascial arterial plexus,https://purl.bioontology.org/ontology/OFL/OFLI...,Arterial plexus of the integument,http://purl.obolibrary.org/obo/FMA_5902
4,https://purl.bioontology.org/ontology/OFL/OFLI...,Arterial plexus of the integument,https://purl.bioontology.org/ontology/OFL/OFLI...,Arterial plexus of the integument,http://purl.obolibrary.org/obo/FMA_5902
...,...,...,...,...,...
868,https://purl.bioontology.org/ontology/OFL/OFLI...,Branch of popliteal artery to soleus muscle,https://purl.bioontology.org/ontology/OFL/OFLI...,Subdivision of popliteal artery,http://purl.obolibrary.org/obo/FMA_69458
869,https://purl.bioontology.org/ontology/OFL/OFLI...,Subdivision of popliteal artery,https://purl.bioontology.org/ontology/OFL/OFLI...,Subdivision of popliteal artery,http://purl.obolibrary.org/obo/FMA_69458
870,https://purl.bioontology.org/ontology/OFL/OFLI...,Medial sural artery,https://purl.bioontology.org/ontology/OFL/OFLI...,Subdivision of popliteal artery,http://purl.obolibrary.org/obo/FMA_69458
871,https://purl.bioontology.org/ontology/OFL/OFLI...,Lateral sural artery,https://purl.bioontology.org/ontology/OFL/OFLI...,Subdivision of popliteal artery,http://purl.obolibrary.org/obo/FMA_69458


In [6]:
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"Wrote {len(df)} rows to {OUTPUT_CSV.name}")

Wrote 873 rows to ofl_2.1.0_subclasses_of_result_fma_superclasses.csv
